# Train RND: churn from DAC MVP

Ноутбук максимально приближен к исходному `train_rnd.ipynb`, но адаптирован под новую задачу `target_churn_from_dac`: OOF-CV, CatBoost, калибровка, расширенные метрики, артефакты, MLflow и data report.

# 0. Environment

In [ ]:
# Если пакет не установлен в kernel, добавляем локальный src в sys.path
import sys
from pathlib import Path

project_root = Path('/Users/underplums/Documents/work/organic-return-dac')
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

project_root, src_path

In [ ]:
# Если используешь .env, раскомментируй
# %load_ext dotenv
# %dotenv /Users/underplums/Documents/work/organic-return-dac/.env

# 1. Imports

In [ ]:
import os
from pathlib import Path
import joblib
from collections import defaultdict
from tqdm.auto import tqdm

import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

import magpie.sql_utils as su

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance

import shap

from sklearn.metrics import (
    average_precision_score as sk_average_precision_score,
    roc_auc_score as sk_roc_auc_score,
    roc_curve,
    precision_recall_curve,
    PrecisionRecallDisplay,
    confusion_matrix as sk_confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    brier_score_loss as sk_brier_score_loss,
)

from cvm_ml_metrics.classification import (
    roc_auc_score,
    precision_recall_auc,
    confusion_matrix,
    log_loss,
    recall_score,
    precision_score,
    sensitivity_specificity,
    balanced_accuracy,
    f1_score,
    fbeta_score,
    brier_score_loss,
    youden_j,
    markedness,
    lift,
    gini,
    ks_stat_bin_class,
    ece_mce_fast,
    matthews_corrcoef,
    cohen_kappa_score,
)

import mlflow
from mlflow import MlflowClient
from mlflow.models.signature import infer_signature

In [ ]:
%load_ext autoreload
%autoreload 2

from cvm_model.io import State
import cvm_model.functions as func
import cvm_model.utils as utils
from cvm_model.parameters import (
    RANDOM_STATE,
    template,
    features,
    target,
    score,
    project_name,
    model_type,
    train_data_stat_suffix,
    artifacts_dir,
    jira,
    threshold,
)

Path(artifacts_dir).mkdir(parents=True, exist_ok=True)

event_timestamp = '2026-02-01'
print('target:', target)
print('score:', score)
print('features:', len(features))

# 2. State

In [ ]:
state = State.from_env()
engine = state.credentials.loyalty_gp.sa_engine
session = state.spark.session
s3 = su.get_s3_client()
state, engine, session

# 3. Load Dataset

In [ ]:
# Вариант 1: датасет уже сохранен train/test parquet из preprocess notebook.
# Вариант 2: можно указать один полный parquet в DATASET_PATH.

DATA_DIR = project_root / 'train_test_churn_from_dac'
TRAIN_PATH = DATA_DIR / 'train.parquet'
TEST_PATH = DATA_DIR / 'test.parquet'
FEATURES_PATH = DATA_DIR / 'selected_features.parquet'

DATASET_PATH = project_root / 'df_churn_from_dac.parquet'  # поменяй, если у тебя другой путь
USE_TRAIN_TEST_PARQUET = TRAIN_PATH.exists() and TEST_PATH.exists()

USE_TRAIN_TEST_PARQUET, TRAIN_PATH, TEST_PATH, DATASET_PATH

In [ ]:
if USE_TRAIN_TEST_PARQUET:
    train_df = pd.read_parquet(TRAIN_PATH)
    test_df = pd.read_parquet(TEST_PATH)
    df = train_df.copy().reset_index(drop=True)
    holdout_df = test_df.copy().reset_index(drop=True)
    print('Loaded train/test parquet')
else:
    df = pd.read_parquet(DATASET_PATH).reset_index(drop=True)
    holdout_df = None
    print('Loaded single dataset parquet')

print('df:', df.shape)
if holdout_df is not None:
    print('holdout_df:', holdout_df.shape)

display(df.head())

In [ ]:
assert target in df.columns, f'Нет target-колонки: {target}'
assert 'contact_id' in df.columns, 'Нет contact_id'

df['contact_id'] = df['contact_id'].astype('int64')
if holdout_df is not None:
    holdout_df['contact_id'] = holdout_df['contact_id'].astype('int64')

print('Target distribution:')
display(df[target].value_counts(dropna=False).to_frame('cnt').assign(share=lambda x: x['cnt'] / x['cnt'].sum()))

assert df[target].isna().sum() == 0, 'Есть NA в target'
assert df['contact_id'].nunique() == len(df), 'Есть дубли contact_id в train df'

# 4. DAC Segments

In [ ]:
# Аналог старого segment из train_rnd, но под новую DAC-задачу.
# Если dac_segment_12m уже был создан в preprocess notebook, используем его.
# Иначе строим сегмент из DAC-флагов/количества DAC-месяцев.

def add_dac_segment(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    if 'dac_segment_12m' in data.columns:
        data['segment'] = data['dac_segment_12m'].astype(str)
        return data

    if {'is_stable_dac', 'is_regular_dac', 'is_unstable_dac', 'is_new_dac'} <= set(data.columns):
        data['segment'] = 'other_dac'
        data.loc[data['is_stable_dac'] == 1, 'segment'] = 'stable_dac'
        data.loc[data['is_regular_dac'] == 1, 'segment'] = 'regular_dac'
        data.loc[data['is_unstable_dac'] == 1, 'segment'] = 'unstable_dac'
        data.loc[data['is_new_dac'] == 1, 'segment'] = 'new_dac'
        return data

    if 'dac_months_last_12' in data.columns:
        data['segment'] = pd.cut(
            data['dac_months_last_12'].fillna(0),
            bins=[-1, 0, 1, 5, 9, 12],
            labels=['no_dac_12m', 'new_dac', 'unstable_dac', 'regular_dac', 'stable_dac'],
        ).astype(str)
        return data

    data['segment'] = 'all'
    return data


df = add_dac_segment(df)
if holdout_df is not None:
    holdout_df = add_dac_segment(holdout_df)

df['strat'] = df['segment'].astype(str) + '_' + df[target].astype(str)

# Если есть слишком маленькие страты, fallback на target, иначе StratifiedKFold упадет.
strat_counts = df['strat'].value_counts()
if (strat_counts < 5).any():
    print('Есть маленькие страты, используем только target для CV stratification')
    df['strat'] = df[target].astype(str)

print('Segment distribution:')
display(df['segment'].value_counts(normalize=True).to_frame('share'))

print('Target by segment:')
display(pd.DataFrame(df.groupby('segment')[target].agg(['mean', 'count'])).sort_values('mean', ascending=False))

# 5. Choose Features

In [ ]:
# Берем features из parameters.py, но оставляем только реально доступные колонки.
# Это важно, если preprocess собрал не весь набор или часть колонок была удалена после feature selection.

base_features = list(dict.fromkeys(features))
if FEATURES_PATH.exists():
    selected_from_file = pd.read_parquet(FEATURES_PATH)['feature'].tolist()
    print('Loaded selected_features:', len(selected_from_file))
    # Можно переключить на selected_from_file, если хочешь обучаться только на отобранных фичах:
    # base_features = selected_from_file

missing_features = sorted(set(base_features) - set(df.columns))
features_model = [f for f in base_features if f in df.columns and f not in [target, score, 'contact_id']]

print('Missing features:', len(missing_features))
print(missing_features[:100])
print('Model features:', len(features_model))
pd.DataFrame({'feature': features_model}).head(100)

In [ ]:
# Заполняем recency так же, как в оригинальном train_rnd.
recency_cols = [
    'cheque_recency',
    'login_recency',
    'omni_qr_recency',
    'omni_features_recency',
    'perf_recency',
]
for col in recency_cols:
    if col in df.columns:
        df[col] = df[col].fillna(999)
    if holdout_df is not None and col in holdout_df.columns:
        holdout_df[col] = holdout_df[col].fillna(999)

# CatBoost умеет работать с NaN, поэтому остальные пропуски не имьютим здесь.
# Но проверим картину по пропускам.
na_report = df[features_model].isna().mean().sort_values(ascending=False).to_frame('na_share')
display(na_report.head(30))

# 5.1 Baseline Checklist: Goal And Business Metric

Цель baseline-части: зафиксировать простую воспроизводимую точку сравнения до основного CatBoost-эксперимента.

Бизнес-смысл ошибок:
- `false negative`: клиент реально уйдет из DAC, но модель не пометила его как риск. Возможная потеря DAC-клиента без удерживающего действия.
- `false positive`: клиент не уйдет из DAC, но модель пометила его как риск. Возможный лишний маркетинговый контакт/бюджет.

Для первого MVP основная offline-метрика: `PR-AUC / Average Precision`, потому что таргет несбалансирован. Дополнительно смотрим `ROC-AUC`, `F1/F-beta`, `Recall`, `Precision`, calibration/Brier и качество по DAC-сегментам.

In [ ]:
business_setup = {
    'task_type': 'binary_classification',
    'positive_class': 'target_churn_from_dac = 1',
    'prediction_horizon': 'next calendar month',
    'main_offline_metric': 'PR-AUC / Average Precision',
    'secondary_metrics': ['ROC-AUC', 'F1', 'F0.5', 'F2', 'Recall', 'Precision', 'Brier score'],
    'false_negative_cost': 'DAC churn risk missed',
    'false_positive_cost': 'extra retention/marketing contact',
}
business_setup

# 5.2 Train/Test Split Strategy

Если `train.parquet` и `test.parquet` уже были сохранены после preprocess, используем их как train/holdout. Если есть только один общий датасет, делаем `train_test_split` со стратификацией по `target + DAC segment`.

Основной CatBoost ниже по-прежнему обучается на `df`, как в оригинальном `train_rnd`. Здесь создаются отдельные `baseline_train_df` и `baseline_test_df` для checklist/baseline-экспериментов.

In [ ]:
if holdout_df is not None:
    baseline_train_df = df.copy().reset_index(drop=True)
    baseline_test_df = holdout_df.copy().reset_index(drop=True)
    split_source = 'pre-saved train/test parquet'
else:
    split_strat = df['strat'] if 'strat' in df.columns else df[target].astype(str)
    split_counts = split_strat.value_counts()
    if (split_counts < 2).any():
        split_strat = df[target]

    baseline_train_df, baseline_test_df = train_test_split(
        df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=split_strat,
    )
    baseline_train_df = baseline_train_df.reset_index(drop=True)
    baseline_test_df = baseline_test_df.reset_index(drop=True)
    split_source = 'notebook train_test_split'

print('Split source:', split_source)
print('baseline_train_df:', baseline_train_df.shape)
print('baseline_test_df:', baseline_test_df.shape)

print('Target distribution train:')
display(baseline_train_df[target].value_counts(normalize=True).to_frame('share'))

print('Target distribution test:')
display(baseline_test_df[target].value_counts(normalize=True).to_frame('share'))

overlap = set(baseline_train_df['contact_id']) & set(baseline_test_df['contact_id'])
print('Contact overlap:', len(overlap))
assert len(overlap) == 0, 'Есть пересечение contact_id между baseline train/test'

# 5.3 Simple Features For Baseline Diagnostics

Добавляем несколько простых производных признаков для baseline-экспериментов и анализа ошибок. Они не добавляются автоматически в основной `features_model`, чтобы не менять основной CatBoost-flow без явного решения.

In [ ]:
def add_simple_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()

    if {'dac_months_last_12', 'dac_age_months'} <= set(data.columns):
        data['simple_dac_density_by_age'] = data['dac_months_last_12'] / data['dac_age_months'].replace(0, np.nan)

    if {'trans_count_1', 'trans_count_3'} <= set(data.columns):
        data['simple_trans_1_to_3_ratio'] = data['trans_count_1'] / data['trans_count_3'].replace(0, np.nan)

    if {'login_count_3', 'login_count_6'} <= set(data.columns):
        data['simple_login_3_to_6_ratio'] = data['login_count_3'] / data['login_count_6'].replace(0, np.nan)

    if {'cheque_recency', 'login_recency'} <= set(data.columns):
        data['simple_recency_gap_login_minus_cheque'] = data['login_recency'] - data['cheque_recency']

    return data

baseline_train_df = add_simple_features(baseline_train_df)
baseline_test_df = add_simple_features(baseline_test_df)

simple_features = [c for c in baseline_train_df.columns if c.startswith('simple_')]
baseline_features = list(dict.fromkeys(features_model + simple_features + ['segment']))
baseline_features = [c for c in baseline_features if c in baseline_train_df.columns and c not in [target, score, 'contact_id', 'strat']]

print('simple_features:', simple_features)
print('baseline_features:', len(baseline_features))

# 5.4 Naive Baseline And Pipeline+ColumnTransformer

Делаем два простых baseline:
- `DummyClassifier(strategy='most_frequent')`
- `LogisticRegression` через `Pipeline + ColumnTransformer`: imputer -> scaler -> OHE -> model

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

X_base = baseline_train_df[baseline_features]
y_base = baseline_train_df[target]

categorical_features = X_base.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
numeric_features = [c for c in baseline_features if c not in categorical_features]

print('numeric_features:', len(numeric_features))
print('categorical_features:', categorical_features)

numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', ohe),
])

linear_preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, numeric_features),
        ('cat', categorical_preprocess, categorical_features),
    ],
    remainder='drop',
)

dummy_pipeline = Pipeline(steps=[
    ('preprocess', linear_preprocess),
    ('model', DummyClassifier(strategy='most_frequent')),
])

logistic_pipeline = Pipeline(steps=[
    ('preprocess', linear_preprocess),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
])

# 5.5 Baseline CV: 5 Folds, Mean And Std

Фиксируем среднее качество и стандартное отклонение на 5 фолдах.

In [ ]:
baseline_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scoring = {
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
    'f1': 'f1',
}

baseline_cv_results = {}
for name, pipe in {
    'dummy_most_frequent': dummy_pipeline,
    'logistic_ohe': logistic_pipeline,
}.items():
    print('CV:', name)
    res = cross_validate(
        pipe,
        X_base,
        y_base,
        cv=baseline_cv,
        scoring=cv_scoring,
        n_jobs=-1,
        return_train_score=False,
    )
    baseline_cv_results[name] = res

baseline_cv_report = []
for model_name_, res in baseline_cv_results.items():
    row = {'model': model_name_}
    for metric_name in cv_scoring:
        values = res[f'test_{metric_name}']
        row[f'{metric_name}_mean'] = values.mean()
        row[f'{metric_name}_std'] = values.std()
    baseline_cv_report.append(row)

baseline_cv_report = pd.DataFrame(baseline_cv_report)
display(baseline_cv_report)

# 5.6 Tree Baseline: GBM With Early Stopping

Аналог шага “деревянная модель”: используем `HistGradientBoostingClassifier` как быстрый sklearn-GBM baseline с early stopping. Основная production-модель ниже остается CatBoost, как в оригинальном `train_rnd`.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

try:
    tree_ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    tree_ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

tree_numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

tree_categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', tree_ohe),
])

tree_preprocess = ColumnTransformer(
    transformers=[
        ('num', tree_numeric_preprocess, numeric_features),
        ('cat', tree_categorical_preprocess, categorical_features),
    ],
    remainder='drop',
)

gbm_pipeline = Pipeline(steps=[
    ('preprocess', tree_preprocess),
    ('model', HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=RANDOM_STATE,
    )),
])

print('CV: gbm_early_stopping')
gbm_cv = cross_validate(
    gbm_pipeline,
    X_base,
    y_base,
    cv=baseline_cv,
    scoring=cv_scoring,
    n_jobs=-1,
    return_train_score=False,
)

gbm_cv_report = {'model': 'gbm_early_stopping'}
for metric_name in cv_scoring:
    values = gbm_cv[f'test_{metric_name}']
    gbm_cv_report[f'{metric_name}_mean'] = values.mean()
    gbm_cv_report[f'{metric_name}_std'] = values.std()

gbm_cv_report = pd.DataFrame([gbm_cv_report])
display(pd.concat([baseline_cv_report, gbm_cv_report], ignore_index=True))

# 5.7 Baseline Error Analysis And Slices

Обучаем GBM baseline на train и смотрим holdout: ошибки, децили, DAC-сегменты. Это не заменяет основной CatBoost-анализ ниже, но дает sanity-check до тяжелого эксперимента.

In [ ]:
gbm_pipeline.fit(baseline_train_df[baseline_features], baseline_train_df[target])

baseline_test_df['gbm_score'] = gbm_pipeline.predict_proba(baseline_test_df[baseline_features])[:, 1]
baseline_test_df['gbm_pred_05'] = (baseline_test_df['gbm_score'] >= 0.5).astype(int)

print('GBM holdout PR-AUC:', sk_average_precision_score(baseline_test_df[target], baseline_test_df['gbm_score']))
print('GBM holdout ROC-AUC:', sk_roc_auc_score(baseline_test_df[target], baseline_test_df['gbm_score']))
print(classification_report(baseline_test_df[target], baseline_test_df['gbm_pred_05']))

baseline_test_df['gbm_score_decile'] = pd.qcut(
    baseline_test_df['gbm_score'].rank(method='first'),
    q=10,
    labels=False,
    duplicates='drop',
) + 1

print('Target by score decile:')
decile_report = baseline_test_df.groupby('gbm_score_decile').agg(
    rows=('contact_id', 'count'),
    target_rate=(target, 'mean'),
    score_min=('gbm_score', 'min'),
    score_max=('gbm_score', 'max'),
).sort_index(ascending=False)
display(decile_report)

print('Errors by segment:')
error_report = baseline_test_df.assign(
    is_fp=lambda x: ((x['gbm_pred_05'] == 1) & (x[target] == 0)).astype(int),
    is_fn=lambda x: ((x['gbm_pred_05'] == 0) & (x[target] == 1)).astype(int),
).groupby('segment').agg(
    rows=('contact_id', 'count'),
    target_rate=(target, 'mean'),
    avg_score=('gbm_score', 'mean'),
    fp_rate=('is_fp', 'mean'),
    fn_rate=('is_fn', 'mean'),
).sort_values('target_rate', ascending=False)
display(error_report)

# 6. Choose Model

In [ ]:
best_params = {
    'learning_rate': 0.030116939055511327,
    'l2_leaf_reg': 7.065796405722955,
    'random_strength': 4.130146073546395,
    'colsample_bylevel': 0.7165756165591122,
    'bootstrap_type': 'MVS',
    'grow_policy': 'Depthwise',
    'subsample': 0.772594198668501,
    'depth': 10,
    'min_data_in_leaf': 2055,
}

# По умолчанию максимально близко к оригинальному train_rnd: best_params не включены.
# Чтобы включить тюнингованные параметры, раскомментируй **best_params ниже.
model = CatBoostClassifier(
    random_state=RANDOM_STATE,
    use_best_model=True,
    metric_period=20,
    eval_metric='PRAUC',
    eval_fraction=0.1,
    early_stopping_rounds=50,
    thread_count=-1,
    verbose=-1,
    # auto_class_weights='Balanced',
    # **best_params,
)
model

# 7. Train With Crossvalidation

In [ ]:
# Вариант как в оригинальном train_rnd: OOF predict_proba через cross_val_predict.
# Если CatBoost/окружение начнет капризничать на cross_val_predict, используй следующую ячейку с ручным CV.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

df[['class_0', score]] = cross_val_predict(
    model,
    df[features_model],
    df[target],
    cv=skf.split(df[features_model], df['strat']),
    method='predict_proba',
)

print(df[[target, score]].head())
print('OOF PR-AUC:', sk_average_precision_score(df[target], df[score]))

In [ ]:
# Альтернативный/диагностический CV как в оригинальном ноутбуке: считаем AP gain по фолдам.
# Если предыдущая ячейка уже отработала, эту можно запускать для контроля; score будет перезаписан ручными OOF-предиктами.

metrics_cv = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X = df[features_model]
y = df[target]
z = df['strat']

predicts = np.zeros(len(df))

for i, (train_idx, val_idx) in enumerate(skf.split(X, z), start=1):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    print()
    print(f'Training fold {i}...')
    fold_model = CatBoostClassifier(**model.get_params())
    fold_model.fit(X_train, y_train, plot=False, verbose=0)

    print(f'Scoring fold {i}...')
    predicts[val_idx] = fold_model.predict_proba(X_val)[:, 1]

    ap_gain = sk_average_precision_score(y_val, predicts[val_idx]) - y_val.mean()
    metrics_cv.append(ap_gain)
    print(f'Fold {i} complete. AP gain = {ap_gain:.6f}')

print('AP gain by folds:', metrics_cv)
print('AP gain mean:', np.mean(metrics_cv), 'std:', np.std(metrics_cv))

df[score] = predicts

# 8. Train Final Model And Score Holdout

In [ ]:
model.fit(df[features_model], df[target], plot=False, verbose=1)

df[f'{score}_full_model'] = model.predict_proba(df[features_model])[:, 1]

if holdout_df is not None:
    holdout_df[score] = model.predict_proba(holdout_df[features_model])[:, 1]
    print('Holdout scored:', holdout_df.shape)
    display(holdout_df[[target, score]].head())

In [ ]:
model_path = project_root / 'final_model_churn_from_dac.cbm'
model.save_model(str(model_path))
print(model_path)

# 9. Calibrate With Crossvalidation

In [ ]:
# Калибровка на OOF-score, как в оригинальном train_rnd.
platt = LogisticRegression(C=1e10)  # C=1e10 почти отключает регуляризацию
platt_spline = make_pipeline(SplineTransformer(n_knots=5, degree=3), LogisticRegression(C=1.0))
isotonic = IsotonicRegression(out_of_bounds='clip')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_score = pd.DataFrame(df[score])
y = df[target]
z = df['strat']

X_scored = pd.DataFrame(index=df.index)

for i, (train_idx, val_idx) in enumerate(skf.split(X_score, z), start=1):
    X_train, y_train = X_score.iloc[train_idx], y.iloc[train_idx]
    X_val = X_score.iloc[val_idx]

    print(f'Calibrating on fold {i}...')
    platt.fit(X_train, y_train)
    platt_spline.fit(X_train, y_train)
    isotonic.fit(X_train, y_train)

    X_scored.loc[X_val.index, f'{score}_platt'] = platt.predict_proba(X_val)[:, 1]
    X_scored.loc[X_val.index, f'{score}_platt_spline'] = platt_spline.predict_proba(X_val)[:, 1]
    X_scored.loc[X_val.index, f'{score}_isotonic'] = isotonic.transform(X_val[score])

df[[f'{score}_platt', f'{score}_platt_spline', f'{score}_isotonic']] = X_scored[[f'{score}_platt', f'{score}_platt_spline', f'{score}_isotonic']]
df[[score, f'{score}_platt', f'{score}_platt_spline', f'{score}_isotonic']].describe()

# 10. Evaluate Probabilities

In [ ]:
plt.rcParams['figure.figsize'] = [8, 6]
func.plot_precision_recall_curve(df[target], df[score])

In [ ]:
func.plot_roc(df[target], df[score])

In [ ]:
plt.figure(figsize=(12, 8))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', alpha=0.5)

models_to_plot = [
    (df[score], 'Raw', 'blue'),
    (df[f'{score}_platt'], 'Platt', 'green'),
    (df[f'{score}_platt_spline'], 'Platt Spline', 'purple'),
    (df[f'{score}_isotonic'], 'Isotonic', 'orange'),
]

for preds, label, color in models_to_plot:
    prob_true, prob_pred = calibration_curve(df[target], preds, n_bins=10, strategy='quantile')
    plt.plot(prob_pred, prob_true, marker='o', label=label, color=color)

plt.xlabel('Mean predicted probability')
plt.ylabel('Observed target rate')
plt.title('Calibration Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df[df[target] == 0][score], label='No churn from DAC', kde=True, stat='density', color='blue')
sns.histplot(df[df[target] == 1][score], label='Churn from DAC', kde=True, stat='density', alpha=0.25, color='orange')
plt.title('Labeled Score Histogram')
plt.legend();

In [ ]:
if holdout_df is not None:
    print('Holdout PR-AUC:', sk_average_precision_score(holdout_df[target], holdout_df[score]))
    print('Holdout ROC-AUC:', sk_roc_auc_score(holdout_df[target], holdout_df[score]))
    func.plot_precision_recall_curve(holdout_df[target], holdout_df[score], title_suffix='Holdout')

# 11. Evaluate Classes And Threshold

In [ ]:
def find_threshold(y_true, y_pred, eval_function, **kwargs):
    score_by_threshold = {}
    for t in np.arange(0.05, 1, 0.05):
        y_class = y_pred > t
        score_by_threshold[t] = eval_function(y_true, y_class, **kwargs)
    return max(score_by_threshold.items(), key=lambda x: x[1])[0]


t = df[target]
s = df[score]

for beta in np.arange(0.05, 1, 0.05):
    best_t = find_threshold(t, s, fbeta_score, beta=beta)
    c_beta = s > best_t
    print(round(beta, 2), 'threshold:', round(best_t, 3), 'predicted_positive_share:', round(c_beta.mean(), 4))

In [ ]:
# Основной порог. Можно оставить threshold из parameters.py или подобрать по F1/F0.5.
threshold_f1 = find_threshold(t, s, fbeta_score, beta=1)
threshold_f05 = find_threshold(t, s, fbeta_score, beta=0.5)
threshold_f035 = find_threshold(t, s, fbeta_score, beta=0.35)

print('threshold from parameters.py:', threshold)
print('threshold F1:', threshold_f1)
print('threshold F0.5:', threshold_f05)
print('threshold F0.35:', threshold_f035)

selected_threshold = threshold_f1
c = s > selected_threshold
print(classification_report(t, c))

In [ ]:
metrics = {
    'ROC-AUC': roc_auc_score(t, s),
    'PR-AUC': precision_recall_auc(t, s),
    'AP gain': sk_average_precision_score(t, s) - t.mean(),
    'Log loss': log_loss(t, s),
    'Recall': recall_score(t, c),
    'Precision': precision_score(t, c),
    'Specificity': sensitivity_specificity(t, c)[1],
    'Balanced accuracy': balanced_accuracy(t, c),
    'F1': f1_score(t, c),
    'F0.5': fbeta_score(t, c, beta=0.5),
    'F2': fbeta_score(t, c, beta=2),
    'Brier score': brier_score_loss(t, s),
    'Youden j': youden_j(t, c),
    'Markedness': markedness(t, s),
    'Lift': lift(t, s),
    'Gini': gini(t, s),
    'Kolmogorov-Smirnov statistic': ks_stat_bin_class(t, s),
    'Expected Calibration Error': ece_mce_fast(t, s)[0],
    'Maximum Calibration Error': ece_mce_fast(t, s)[1],
    'Matthews corrcoef': matthews_corrcoef(t, c),
    'Cohen kappa score': cohen_kappa_score(t, c),
}

pd.DataFrame(metrics, index=['value']).T

In [ ]:
cm = confusion_matrix(t, c)
class_labels = ['No churn from DAC', 'Churn from DAC']
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title('Confusion Matrix');

In [ ]:
print('Metrics by DAC segment:')
rows = []
for segment, part in df.groupby('segment'):
    if part[target].nunique() < 2:
        continue
    rows.append({
        'segment': segment,
        'rows': len(part),
        'target_rate': part[target].mean(),
        'roc_auc': sk_roc_auc_score(part[target], part[score]),
        'pr_auc': sk_average_precision_score(part[target], part[score]),
        'ap_gain': sk_average_precision_score(part[target], part[score]) - part[target].mean(),
    })

pd.DataFrame(rows).sort_values('pr_auc', ascending=False)

# 12. Save Artifacts

In [ ]:
# Чистим папку с артефактами
for file in Path(artifacts_dir).iterdir():
    if file.is_file():
        try:
            file.unlink()
        except Exception as e:
            print(f'Failed to delete {file}. Reason: {e}')

# Precision-Recall Curve
func.plot_precision_recall_curve(t, s, artifacts_dir=artifacts_dir)

# ROC
func.plot_roc(t, s, artifacts_dir=artifacts_dir)

# Labeled Score Histogram
plt.figure(figsize=(8, 6))
sns.histplot(s[t == 0], label='No churn from DAC', kde=True, stat='density', color='blue')
sns.histplot(s[t == 1], label='Churn from DAC', kde=True, stat='density', alpha=0.25, color='orange')
title = 'Labeled Score Histogram'
plt.title(title)
plt.legend()
plt.savefig(os.path.join(artifacts_dir, title), bbox_inches='tight')
plt.close()

# Confusion Matrix
cm = confusion_matrix(t, c)
class_labels = ['No churn from DAC', 'Churn from DAC']
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(cmap=plt.cm.Blues, values_format='d')
title = 'Confusion Matrix'
plt.title(title)
plt.savefig(os.path.join(artifacts_dir, title), bbox_inches='tight')
plt.close()

# Features
Path(os.path.join(artifacts_dir, 'features.txt')).write_text('\n'.join(features_model), encoding='utf-8')

print('Artifacts dir:', Path(artifacts_dir).resolve())

In [ ]:
# SHAP summary. На больших данных ограничиваем sample, чтобы ноутбук не умер по памяти.
shap_sample_size = min(500_000, len(df))
explainer = shap.TreeExplainer(model)
sample = df[features_model].sample(shap_sample_size, random_state=RANDOM_STATE)
shap_vals_all = explainer.shap_values(sample)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals_all, sample, show=False)
title = 'SHAP Summary Plot'
plt.title(title)
plt.savefig(os.path.join(artifacts_dir, title), bbox_inches='tight')
plt.show()

# 13. Log Train Run In MLflow

In [ ]:
# Model signature
signature = infer_signature(df.head(1)[features_model], df.head(1)[score])

# Run description
# event_timestamp можно поменять на дату обучающего среза.
description = f'train_{event_timestamp}'

params = {
    'event_timestamp': event_timestamp,
    'threshold': selected_threshold,
    'default_threshold': threshold,
    'features_count': len(features_model),
    'rows_count': len(df),
    'target_rate': float(df[target].mean()),
}
params

In [ ]:
model_name = f'{project_name}_{model_type}'
experiment_name = f'{model_name}_{jira}_train'

logging.info(experiment_name)
logging.info(model_name)
mlflow.set_experiment(experiment_name)
experiment_name, model_name

In [ ]:
with mlflow.start_run(run_name=description, description=description):
    for k, v in params.items():
        mlflow.log_param(k, v)

    for k, v in metrics.items():
        mlflow.log_metric(k, float(v))

    for root, dirs, files in os.walk(artifacts_dir):
        dirs[:] = [d for d in dirs if d != '.ipynb_checkpoints']
        for file in files:
            full_path = os.path.join(root, file)
            mlflow.log_artifact(full_path)

    mlflow.sklearn.log_model(
        model,
        name=model_name,
        signature=signature,
        registered_model_name=model_name,
        tags=params,
    )

client = MlflowClient()
last_version = client.search_model_versions(f"name = '{model_name}'", order_by=['version_number DESC'])[0].version
for k, v in params.items():
    client.set_model_version_tag(name=model_name, version=last_version, key=k, value=v)

last_version

# 14. Feature Importance & Finetuning Diagnostics

In [ ]:
X_train, X_val, y_train, y_val, z_train, _ = train_test_split(
    df[features_model],
    df[target],
    df['strat'],
    stratify=df['strat'],
    test_size=0.3,
    random_state=RANDOM_STATE,
)

model.fit(X_train, y_train, plot=False, verbose=1)
predicts_val = model.predict_proba(X_val)[:, 1]

print('Val PR-AUC:', sk_average_precision_score(y_val, predicts_val))
print('Val ROC-AUC:', sk_roc_auc_score(y_val, predicts_val))

In [ ]:
# Shapley on validation sample
shap_val_sample_size = min(200_000, len(X_val))
X_val_sample = X_val.sample(shap_val_sample_size, random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(model)
shap_vals_val = explainer.shap_values(X_val_sample)
shap.summary_plot(shap_vals_val, X_val_sample)

In [ ]:
# Redundant features, как в оригинальном train_rnd.
reduntant_filter, pairs = func.get_redundant_features(X_train, threshold=0.85)
print('Redundant features:', len(reduntant_filter))
display(pd.DataFrame(pairs, columns=['feature_1', 'feature_2', 'corr']).head(100))

joblib.dump(reduntant_filter, project_root / 'reduntant_filter_churn_from_dac.pkl')

In [ ]:
# Permutation importance. На большом X_val может выполняться долго.
RUN_PERMUTATION_IMPORTANCE = False

if RUN_PERMUTATION_IMPORTANCE:
    r = permutation_importance(
        model,
        X_val,
        y_val,
        scoring='average_precision',
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    pi_df = pd.DataFrame({
        'feature': X_val.columns,
        'importance_mean': r.importances_mean,
        'importance_std': r.importances_std,
    }).sort_values('importance_mean', ascending=False)
    display(pi_df.head(100))

    pi_filter = pi_df.loc[pi_df['importance_mean'] < 0.0001, 'feature'].tolist()
    joblib.dump(pi_filter, project_root / 'pi_filter_churn_from_dac.pkl')

In [ ]:
# Interactions
interactions = model.get_feature_importance(type='Interaction')
interaction_df = pd.DataFrame(interactions, columns=['f1', 'f2', 'score'])
interaction_df['f1_name'] = interaction_df['f1'].apply(lambda x: features_model[int(x)])
interaction_df['f2_name'] = interaction_df['f2'].apply(lambda x: features_model[int(x)])
top_interactions = interaction_df.sort_values(by='score', ascending=False).head(15)
display(top_interactions[['f1_name', 'f2_name', 'score']])

# 15. Save Train Dataset Stats To S3

In [ ]:
train_data_stat_prefix = state.settings.get_prefix(temp=False, suffix=train_data_stat_suffix)
train_data_stat_bucket = train_data_stat_prefix.split('//')[1].split('/')[0]
train_data_stat_prefix = '/'.join(train_data_stat_prefix.split('//')[1].split('/')[1:])
train_data_stat_bucket, train_data_stat_prefix

In [ ]:
report_cols = features_model + [target, score]
logging.info('Saving train dataset stats...')
data_stat_prefix = f'{train_data_stat_prefix}/{event_timestamp}/'
logging.info(data_stat_prefix)
report = func.get_data_report(df, report_cols)

print(report.shape)
display(report.head())

In [ ]:
# Раскомментируй, когда отчет проверен и можно писать в S3.
# su.save_to_s3(report, data_stat_prefix, input_type='df', bucket=train_data_stat_bucket)

# 16. Save Scored Dataset

In [ ]:
scored_path = project_root / f'df_churn_from_dac_scored_{event_timestamp}.parquet'
df.to_parquet(scored_path, index=False)
print(scored_path)

if holdout_df is not None:
    holdout_scored_path = project_root / f'holdout_churn_from_dac_scored_{event_timestamp}.parquet'
    holdout_df.to_parquet(holdout_scored_path, index=False)
    print(holdout_scored_path)

# 17. Save Baseline Pipeline And README

Сохраняем sklearn baseline pipeline отдельно от CatBoost-модели. Это нужно как воспроизводимый baseline artifact: preprocessing + model в одном `pipeline.pkl`.

In [ ]:
pipeline_path = project_root / 'pipeline_churn_from_dac_baseline.pkl'
joblib.dump(gbm_pipeline, pipeline_path)

readme_path = project_root / 'README_churn_from_dac_experiment.md'

final_metrics_for_readme = metrics if 'metrics' in globals() else {}
cv_table_for_readme = pd.concat([baseline_cv_report, gbm_cv_report], ignore_index=True)

readme_text = f'''# Churn From DAC Baseline Experiment

Дата среза / event_timestamp: `{event_timestamp}`

## Цель
Предсказать вероятность того, что клиент, являющийся DAC в базовом месяце, уйдет из DAC в следующем месяце.

## Target
`{target}` = 1, если клиент был DAC в base month и не является DAC в target month.

## Main Metric
Основная offline-метрика для baseline: PR-AUC / Average Precision. Дополнительно смотрим ROC-AUC, F-beta, Recall, Precision, Brier/calibration.

## Dataset
Train rows: `{len(df)}`
Target rate train: `{df[target].mean():.6f}`
Features count: `{len(features_model)}`

## Baseline CV
{cv_table_for_readme.to_markdown(index=False)}

## CatBoost Metrics
{pd.DataFrame(final_metrics_for_readme, index=['value']).T.to_markdown() if final_metrics_for_readme else 'CatBoost metrics are not calculated yet.'}

## Artifacts
- sklearn baseline pipeline: `{pipeline_path}`
- CatBoost model: `{model_path if 'model_path' in globals() else 'not saved yet'}`
- artifacts dir: `{Path(artifacts_dir).resolve()}`

## Risks And Limitations
- Важно контролировать leakage: признаки должны использовать только данные до target month.
- DAC-сегменты сильно связаны с target, поэтому их нужно мониторить отдельно.
- Offline-метрики не заменяют бизнес-оценку эффекта; финальный выбор порога должен учитывать стоимость FP/FN.
- Вероятности требуют калибровки, если дальше используются как абсолютная вероятность, а не только как ранжирование.
'''

readme_path.write_text(readme_text, encoding='utf-8')

print('Saved pipeline:', pipeline_path)
print('Saved README:', readme_path)

# 18. Final Experiment Report

Короткий отчет по первому эксперименту: метрики, важности/SHAP, срезы и ограничения.

In [ ]:
experiment_report = {
    'event_timestamp': event_timestamp,
    'target': target,
    'rows': len(df),
    'target_rate': df[target].mean(),
    'features_count': len(features_model),
    'selected_threshold': selected_threshold if 'selected_threshold' in globals() else None,
    'baseline_best_pr_auc': float(pd.concat([baseline_cv_report, gbm_cv_report], ignore_index=True)['pr_auc_mean'].max()),
    'catboost_pr_auc': float(metrics['PR-AUC']) if 'metrics' in globals() else None,
    'catboost_roc_auc': float(metrics['ROC-AUC']) if 'metrics' in globals() else None,
}

print('Experiment report:')
display(pd.DataFrame(experiment_report, index=['value']).T)

print('Top CatBoost feature importances:')
try:
    fi = pd.DataFrame({
        'feature': features_model,
        'importance': model.get_feature_importance(),
    }).sort_values('importance', ascending=False)
    display(fi.head(30))
except Exception as e:
    print('Feature importance is not available yet:', e)

print('Main risks / checks:')
for item in [
    'Проверить, что все SQL-фичи считаются строго до target_month.',
    'Сравнить качество по DAC-сегментам, а не только overall.',
    'Проверить calibration curve перед использованием score как вероятности.',
    'Подобрать threshold под бизнес-стоимость FP/FN.',
    'Повторить эксперимент на полном датасете после теста на sample.',
]:
    print('-', item)